# CMSC 320 Final Project — Checkpoint 2

**Members:** Pranav Samuel, Aarav Nirmal, Manan Rajput  

## Project Topic  
Analyzing Factors That Contribute to Movie Success  

## Datasets  
- TMDB 5000 Movie Metadata (Kaggle)  
- Worldwide Box Office Rankings (Kaggle)  

## Objective  
The goal of this project is to explore which factors (such as budget, genre, popularity, and ratings) most strongly influence a movie’s financial and critical success.

Adjusted Revenue, Release Dates (77-Present), Budget, Cast, Genre, Keywords, Popularity Ratings, Runtime, Average Rating, Crew (Director, Composer, GFX Designer)

We intend to run a meta-analysis on movie data 

Tests We Plan To Run: 

1. Production Efficiency (Chi-Square)
    Type: Chi-Squared Test of Independence
    - We will categorize movies into "High Budget" vs. "Low Budget" (using the median) and "Flop" vs. "Success" (Revenue < Budget vs. Revenue > Budget).
    - This test answers the question, "Does spending more money statistically decrease your risk of failing, or are low-budget movies actually safer bets?"
    - $H_0$: There is no association between budget size (High vs. Low) and financial outcome (Success vs. Flop). They are independent.
    - $H_A$: There is a significant association between budget size and financial outcome. (e.g., High-budget movies are statistically more likely to be successes than low-budget movies).
    - The plot we will use is the Mosaic Plot


2. Artistic Effects: Director-Writer Overlap
    Type: Comparison of Means (Two-Sample t-test)
    - We will create two groups: Group A (Movies where the Director is also a Writer) and Group B (Movies where they are different people). We then compare their Average Ratings.
    - This test answers: "Does a singular idea lead to a statistically significant increase in movie quality, or is it better to have specialists?"
    - $H_0$: The mean Average Rating for movies with a Director-Writer overlap is equal to the mean Average Rating for movies with separate specialists ($\mu_1 = \mu_2$).
    - $H_A$: The mean Average Rating for movies with a Director-Writer overlap is significantly different from movies with separate specialists ($\mu_1 \neq \mu_2$).
     - The plot we will use is the Box Plot with points 

3. Market Analysis: ANOVA (5 Genres vs. Box Office Revenue)
    Type: Analysis of Variance (Multiple Group Comparison)
    - We take the big three distinct genre buckets (High-Energy, Suspenseful, Stylized, Emotional, and Lighthearted) and compare their Worldwide Revenue.
    - This test answers: "Is there a premier genre that consistently outperforms the others, or is the financial difference between genres just a matter of chance?"
    - $H_0$: The mean Worldwide Revenue is the same across all three genre groups ($\mu_{High-Energy} = \mu_{Suspenseful} = \mu_{Stylized} = \mu_{Emotional} = \mu_{Lighthearted}$).
    - $H_A$: At least one genre group has a mean Worldwide Revenue that is significantly different from the others.
    - The plot we will use is the Bar Chart with Error Bars



Needs (Data):
1. Worldwide Revenue (Adj.) ✔️
2. Budget (Adj.) ✔️
3. Rating Data  ✔️
4. Crew (Writing) ✔️
5. Director ✔️
6. Genre Data (List) ✔️ 

1. (1, 2) -> Revenue and Budget
2. (3, 4, 5) -> Rating, Crew, and Director
3. (1, 6) -> Revenue and Genre Data

Columns for Combined CSV: 
1. Revenue(Adj) (Integer)
2. Budget(Adj) (Integer)
3. Rating Data (Double)
4. Crew (String List)
5. Director (String)
6. Genre Data (String)

Columns to Add for Analysis:
7. Profit Margin 
8. Director in Writing Crew? (Boolean)

## Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
# other imports

## Data Loading

In this section, we import the datasets and inspect their structure.

In [ ]:
tmdb_movies = pd.read_csv("tmdb_5000_movies.csv")
tmdb_credits = pd.read_csv("tmdb_5000_credits.csv")

cpi = pd.read_csv("CORESTICKM159SFRBATL.csv")

## Data Preprocessing

In this section, we are going to clean and setup the data for analysis.

First, we are going to look into our revenues and adjust them according to inflation with out CPI dataset.

In [ ]:
# setting up cpi for grouping by year
cpi.columns = ["date", "cpi"]
cpi['date'] = pd.to_datetime(cpi['date'])
cpi['year'] = cpi['date'].dt.year
cpi['cpi'] = pd.to_numeric(cpi['cpi'], errors='coerce')

yearly_cpi = cpi.groupby('year')['cpi'].mean().reset_index()
display(yearly_cpi)

We are applying each CPI to every movie in our box office database and combining them together for easier parsing.

In [ ]:
# list all our files
movie_files = glob.glob("ranking_summary_*.csv")

adjusted_movies_list = []
latest_cpi = yearly_cpi['cpi'].iloc[-1]

# apply cpi to each file
for file in movie_files:
    df = pd.read_csv(file)
    
    # get the year from title
    year_from_file = int(file.split('_')[2].split('.')[0])
    df['year'] = year_from_file

    df = df.merge(yearly_cpi, on='year', how='left')
    
    # adjust for inflation
    df['worldwide'] = df['worldwide'].replace('[\$,]', '', regex=True)
    df['worldwide'] = pd.to_numeric(df['worldwide'], errors='coerce')
    df['worldwide_adjusted'] = df['worldwide'] * (latest_cpi / df['cpi'])
    
    # add each df to combine
    adjusted_movies_list.append(df)

combined_movies = pd.concat(adjusted_movies_list, ignore_index=True)
combined_movies.head()

We are merging the two TMDB sets to have a more complete dataset. We also dropped columns that are not going to contribute to our testing.

In [ ]:
# merging our two tmdb datasets
tmdb_movies['title'] = tmdb_movies['title'].astype(str)
tmdb_credits['title'] = tmdb_credits['title'].astype(str)

tmdb_full = pd.merge(tmdb_movies, tmdb_credits, on='title', how='inner')

# dropping unnecessary columns
tmdb_full = tmdb_full.drop(columns=['id', 'tagline', 'status','overview', 'keywords', 'homepage', 'original_title', 'popularity', 'production_companies', 'production_countries', 'revenue', 'cast', 'spoken_languages'])
tmdb_full.head()

We rearranged rows and adjusted for budget inflation (similar to process for worldwide revenue adjustment).

In [ ]:
# adjust for budget with cpi
tmdb_full['release_date'] = pd.to_datetime(tmdb_full['release_date'])
tmdb_full['year'] = tmdb_full['release_date'].dt.year

tmdb_full = tmdb_full.merge(yearly_cpi, on='year', how='left')
tmdb_full['budget'] = pd.to_numeric(tmdb_full['budget'], errors='coerce')

tmdb_full['budget_adjusted'] = tmdb_full['budget'] * (latest_cpi / tmdb_full['cpi'])

# dropping cpi
tmdb_full = tmdb_full.drop(columns=['cpi', 'budget'])

# rearranging rows
tmdb_full = tmdb_full[['movie_id', 'title', 'genres','original_language', 'year', 'release_date', 'budget_adjusted','crew', 'vote_average', "vote_count"]]
tmdb_full

We are merging the two datasets we have (tmdb_full and combined_movies). This will give us a holistic form of analysis of the movies. There are however certain issues with merging, for example movies which may not match (only in one of the databases) as well as having overlapping data. We overcome these issues by only choosing to take movies which are properly represented in both databases as well as taking priority of revenue information in the combined_movies database. 

In [ ]:
tmdb_full['title'] = tmdb_full['title'].astype(str).str.strip()
combined_movies['title'] = combined_movies['title'].astype(str).str.strip()

merged_movies = pd.merge(
    tmdb_full,
    combined_movies,
    on='title',
    how='inner'
)
merged_movies
